# VRB Lab batch launch (stress test)

Uses BinderHub's build/launch API to **launch the current repo (`gh/yxzhan/ebim-vrb-lab`) concurrently, in bulk** on `ebim-binder.aicor.dev`. Anonymous, no auth, no TLS verification.

BinderHub's `/build/<provider>/<spec>` is a **Server-Sent Events (SSE)** endpoint; one call handles *build + launch* together — you must keep the connection open until `ready` to receive the server URL + token.

> ⚠️ This really launches N containers on the target hub. Start with a small `N_LAUNCHES`, then increase. Anonymous BinderHub usually assigns a **separate temporary user** per connection, so N launches = N independent servers (if the hub dedupes by account, it reuses one).

In [1]:
# ================== Config ==================
BINDER_HOST = "https://ebim-binder.aicor.dev"   # target hub
PROVIDER    = "gh"                                # github
SPEC        = "yxzhan/ebim-vrb-lab/main"       # <user>/<repo>/<ref>; ref may be a branch name or a full commit SHA
                                                  # a SHA is more deterministic, e.g. ".../ebim-vrb-lab/<40-char-sha>"

N_LAUNCHES  = 20          # how many to launch in total
CONCURRENCY = 10           # parallelism: how many at once (BinderHub RateLimiter relaxed, so parallel is fine)

PER_LAUNCH_TIMEOUT = 900 # max seconds to wait per launch (first build can be slow; fast once the image is built)
VERIFY_TLS  = False      # do not verify the TLS certificate
AUTH_TOKEN  = None        # no auth needed; if the hub requires it, put a JupyterHub API token string here
ADMIN_TOKEN = None        # for stopping pods: a hub admin API token. Leave None to use each server's own token
# ==========================================

In [2]:
# ===== node -> geographic region =====
# a k8s node name looks like "l4-vm-283.c.ebim26ham-283.internal"; take the first segment (instance name) to match.
NODE_ZONES = {
    # instance     : (zone,              static egress IP)
    "l4-vm-281": ("europe-west4-c", "<static-ip-281>"),
    "l4-vm-282": ("us-central1-c",  "<static-ip-282>"),
    "l4-vm-283": ("us-west4-c",     "<static-ip-283>"),
    "l4-vm-284": ("europe-west4-c", "<static-ip-284>"),
    "l4-vm-285": ("us-east1-b",     "<static-ip-285>"),
    "l4-vm-286": ("us-east1-b",     "<static-ip-286>"),
    "l4-vm-287": ("us-west4-c",     "<static-ip-287>"),
    "l4-vm-288": ("europe-west4-c", "<static-ip-288>"),
    "l4-vm-289": ("europe-west4-c", "<static-ip-289>"),
    "l4-vm-290": ("europe-west4-c", "<static-ip-290>"),
}

def node_instance(node):
    """take the instance name from a full node name: l4-vm-283.c.ebim26ham-283.internal -> l4-vm-283"""
    return node.split(".", 1)[0] if node else None

def node_zone(node):
    """return the node's zone, or '?' if unknown."""
    inst = node_instance(node)
    return NODE_ZONES.get(inst, ("?", None))[0] if inst else "?"

In [3]:
import json, re, time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

if not VERIFY_TLS:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BUILD_URL = f"{BINDER_HOST}/build/{PROVIDER}/{SPEC}"

# during 'launching' the k8s events include:
#   [Normal] Successfully assigned binder/jupyter-<user> to l4-vm-283.c.ebim26ham-283.internal
SCHED_RE = re.compile(r"Successfully assigned\s+(?P<ns_pod>\S+)\s+to\s+(?P<node>\S+)")

def launch_one(idx):
    """Open an SSE build/launch stream until ready / failed / timeout. Returns a result dict."""
    headers = {"Accept": "text/event-stream", "Cache-Control": "no-cache"}
    if AUTH_TOKEN:
        headers["Authorization"] = f"token {AUTH_TOKEN}"

    res = {"idx": idx, "ok": False, "phase": None, "phases": [],
           "server_url": None, "token": None, "image": None,
           "node": None, "pod": None, "namespace": None,
           "messages": [], "elapsed": None, "error": None}
    t0 = time.time()
    try:
        with requests.get(BUILD_URL, headers=headers, stream=True,
                          timeout=(10, PER_LAUNCH_TIMEOUT),
                          verify=VERIFY_TLS) as r:
            r.raise_for_status()
            for raw in r.iter_lines(decode_unicode=True):
                if time.time() - t0 > PER_LAUNCH_TIMEOUT:
                    res.update(phase="timeout", error="per-launch timeout")
                    break
                if not raw or not raw.startswith("data:"):
                    continue
                try:
                    ev = json.loads(raw[len("data:"):].strip())
                except json.JSONDecodeError:
                    continue

                msg = (ev.get("message") or "").strip()
                if msg:
                    res["messages"].append(msg)
                    m = SCHED_RE.search(msg)
                    if m and not res["node"]:
                        res["node"] = m.group("node")
                        ns_pod = m.group("ns_pod")
                        if "/" in ns_pod:
                            res["namespace"], res["pod"] = ns_pod.split("/", 1)
                        else:
                            res["pod"] = ns_pod

                ph = ev.get("phase")
                if ph and (not res["phases"] or res["phases"][-1] != ph):
                    res["phases"].append(ph)
                if ph == "ready":
                    res.update(ok=True, phase="ready",
                               server_url=ev.get("url"),
                               token=ev.get("token"),
                               image=ev.get("image"))
                    break
                if ph in ("failed", "failure"):
                    res.update(phase=ph, error=msg)
                    break
    except Exception as e:
        res["error"] = repr(e)
    res["elapsed"] = round(time.time() - t0, 1)
    return res

In [4]:
results = []
t_start = time.time()
print(f"launching {N_LAUNCHES} x  {BUILD_URL}   (parallel, concurrency={CONCURRENCY})\n")

with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
    futs = [ex.submit(launch_one, i) for i in range(N_LAUNCHES)]
    for f in as_completed(futs):
        r = f.result()
        results.append(r)

        tag = "OK " if r["ok"] else "ERR"
        trail = r["server_url"] if r["ok"] else (r["error"] or r["phase"])
        print(f"[{tag}] #{r['idx']:>3}  {r['elapsed']:>6}s  "
              f"node={r['node'] or '?'} ({node_zone(r['node'])})  "
              f"{' -> '.join(r['phases'])}  {trail}")

ok = sum(r["ok"] for r in results)
rl = sum("rate limit" in (r["error"] or "").lower() for r in results)
print(f"\n==== done: ok={ok}/{len(results)}  wall={round(time.time()-t_start,1)}s ====")
if rl:
    print(f"!! {rl} of them hit the RateLimiter (429): make sure "
          f"c.RateLimiter.limit is large enough before raising CONCURRENCY/N")

launching 20 x  https://ebim-binder.aicor.dev/build/gh/yxzhan/ebim-vrb-lab/main   (parallel, concurrency=10)

[OK ] #  3    12.7s  node=l4-vm-289.c.ebim26ham-289.internal (europe-west4-c)  built -> launching -> ready  https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-zzv9ww3x/
[OK ] #  0    13.0s  node=l4-vm-290.c.ebim26ham-290.internal (europe-west4-c)  built -> launching -> ready  https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-pozm0srl/
[OK ] #  6    13.2s  node=l4-vm-290.c.ebim26ham-290.internal (europe-west4-c)  built -> launching -> ready  https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-3yjjl30q/
[OK ] #  4    14.3s  node=l4-vm-284.c.ebim26ham-284.internal (europe-west4-c)  built -> launching -> ready  https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-acllr9yp/
[OK ] #  5    14.4s  node=l4-vm-288.c.ebim26ham-288.internal (europe-west4-c)  built -> launching -> ready  https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-dr1mlxvk/
[OK ] #  1    14.5s 

In [5]:
# which node/zone each lab landed on + distribution
from collections import Counter

print(f"{'#':>3}  {'node':<14} {'zone':<16} {'pod':<45} {'s':>6}")
for r in sorted(results, key=lambda x: x["idx"]):
    inst = node_instance(r["node"]) or "-"
    print(f"{r['idx']:>3}  {inst:<14} {node_zone(r['node']):<16} "
          f"{(r['pod'] or '-'):<45} {r['elapsed']:>6}")

node_dist = Counter(node_instance(r["node"]) or "(unknown)" for r in results)
print("\n---- node distribution ----")
for inst, n in node_dist.most_common():
    print(f"{n:>3} x  {inst:<14} {node_zone(inst)}")

zone_dist = Counter(node_zone(r["node"]) for r in results)
print("\n---- zone distribution ----")
for zone, n in zone_dist.most_common():
    print(f"{n:>3} x  {zone}")
print(f"\n{len(results)} labs across "
      f"{len([k for k in node_dist if k != '(unknown)'])} nodes / "
      f"{len([k for k in zone_dist if k != '?'])} zones")

# save a CSV for later comparison
import csv
with open("stress_nodes_final.csv", "w", newline="") as fp:
    w = csv.writer(fp)
    w.writerow(["idx", "ok", "instance", "zone", "node", "namespace", "pod",
                "server_url", "elapsed_s", "error"])
    for r in sorted(results, key=lambda x: x["idx"]):
        w.writerow([r["idx"], r["ok"], node_instance(r["node"]), node_zone(r["node"]),
                    r["node"], r["namespace"], r["pod"],
                    r["server_url"], r["elapsed"], r["error"]])
print("-> stress_nodes_final.csv")

  #  node           zone             pod                                                s
  0  l4-vm-290      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-pozm0srl            13.0
  1  l4-vm-288      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-uykrmkm7            14.5
  2  l4-vm-282      us-central1-c    jupyter-yxzhan-ebim-vrb-lab-cwxj1qt0            17.8
  3  l4-vm-289      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-zzv9ww3x            12.7
  4  l4-vm-284      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-acllr9yp            14.3
  5  l4-vm-288      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-dr1mlxvk            14.4
  6  l4-vm-290      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-3yjjl30q            13.2
  7  l4-vm-289      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-d1qt17sp            14.5
  8  l4-vm-282      us-central1-c    jupyter-yxzhan-ebim-vrb-lab-bf126g2m            16.3
  9  l4-vm-284      europe-west4-c   jupyter-yxzhan-ebim-vrb-lab-wme4yog1            15.2
 10  l4-vm

In [6]:
# access links for successful launches (root + straight into JupyterLab)
for r in sorted(results, key=lambda x: x["idx"]):
    if not r["ok"]:
        print(f"#{r['idx']:>3}  FAILED  ({r['phase']})  {r['error']}")
        continue
    base = r["server_url"]
    tok  = r.get("token")
    root_link = f"{base}?token={tok}" if tok else base
    lab_link  = f"{base}proxy/8899/?token={tok}" if tok else f"{base}proxy/8899/"
    print(f"[{node_instance(r['node']) or '?'} / {node_zone(r['node'])}] {lab_link}")

[l4-vm-290 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-pozm0srl/proxy/8899/?token=G1yPM5lPSOScX-jGxcSxng
[l4-vm-288 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-uykrmkm7/proxy/8899/?token=XaYKVyrbQui_LuW-dHC19A
[l4-vm-282 / us-central1-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-cwxj1qt0/proxy/8899/?token=OAZOLb7CRjSBYTKQ7lhypA
[l4-vm-289 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-zzv9ww3x/proxy/8899/?token=TbgMVD6PRBC9zOloyUirpw
[l4-vm-284 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-acllr9yp/proxy/8899/?token=21IfTEc_RoW04vDybfbgcA
[l4-vm-288 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-dr1mlxvk/proxy/8899/?token=xH5xASHKQiGoRYs252I8mA
[l4-vm-290 / europe-west4-c] https://ebim-jupyter.aicor.dev/user/yxzhan-ebim-vrb-lab-3yjjl30q/proxy/8899/?token=u5H9BL_pTR6f_CsCaW04JA
[l4-vm-289 / europe-west4-c] https://ebim-jupyter.aicor.

In [7]:
# ===== latency: application-layer RTT via the proxy + ingress-baseline decomposition =====
# the whole cluster has one public ingress (ebim-jupyter); node static IPs are egress-only (not pingable),
# so this is the only measurable path — and it is exactly the path VNC uses. Method:
#   base = you -> ingress RTT (hit /hub/health; the ingress/hub answers directly, not forwarded to a pod)
#   pod  = you -> ingress -> pod RTT (hit each server's api/status)
#   leg  = pod.min - base.min  ~= the ingress -> node mesh hop (cross-region detours show up here)
# take min everywhere = the network floor (filters out server-side jitter from GPU load on the pod).
import statistics
from urllib.parse import urlsplit

LAT_PINGS   = 30
LAT_TIMEOUT = 15

def http_rtts(url, headers):
    """Ping LAT_PINGS times over a persistent connection; return the sorted RTT (ms) list (200s only)."""
    xs = []
    with requests.Session() as s:
        for _ in range(LAT_PINGS):
            try:
                t0 = time.perf_counter()
                resp = s.get(url, headers=headers, timeout=LAT_TIMEOUT,
                             verify=VERIFY_TLS, allow_redirects=False)
                dt = (time.perf_counter() - t0) * 1000
                if resp.status_code < 400:
                    xs.append(dt)
            except Exception:
                pass
    return sorted(xs)

def _min(xs):
    return round(xs[0], 1) if xs else None

ready = [r for r in results if r.get("ok")]
if not ready:
    print("No ready servers; run the launch cell above first.")
else:
    # vantage point
    try:
        myip = requests.get("https://api.ipify.org", timeout=5).text.strip()
        print(f"vantage point: local egress IP = {myip} (your EU network)")
    except Exception:
        print("vantage point: local machine (your EU network)")

    # ingress baseline
    pr = urlsplit(ready[0]["server_url"])
    base_xs = http_rtts(f"{pr.scheme}://{pr.netloc}/hub/health", {})
    base = _min(base_xs)
    if base is not None:
        print(f"you -> ingress ({pr.netloc}) baseline RTT: min={base}ms  "
              f"p50={round(base_xs[len(base_xs)//2],1)}ms\n")
    else:
        print(f"ingress baseline not measurable (/hub/health has no <400 response); leg column will be empty\n")

    print(f"hitting api/status via the proxy x {LAT_PINGS}, taking min:\n")
    print(f"{'#':>3}  {'node':<12} {'zone':<16} {'n':>3}  {'min':>7} {'p95':>7} {'leg~node':>9}")
    rows = []
    for r in sorted(ready, key=lambda x: x["idx"]):
        tok = r.get("token")
        hdr = {"Authorization": f"token {tok}"} if tok else {}
        xs = http_rtts(r["server_url"].rstrip("/") + "/api/status", hdr)
        inst = node_instance(r["node"]) or "-"
        zone = node_zone(r["node"])
        mn = _min(xs)
        p95 = round(xs[min(len(xs) - 1, int(len(xs) * 0.95))], 1) if xs else None
        leg = round(mn - base, 1) if (mn is not None and base is not None) else None
        rows.append({"inst": inst, "zone": zone, "min": mn, "leg": leg})
        print(f"{r['idx']:>3}  {inst:<12} {zone:<16} {len(xs):>3}  "
              f"{str(mn or '-'):>7} {str(p95 or '-'):>7} {str(leg if leg is not None else '-'):>9}")

    # aggregate by zone: the ingress->node leg best shows cross-region detours
    by_zone = {}
    for row in rows:
        if row["leg"] is not None:
            by_zone.setdefault(row["zone"], []).append(row["leg"])
    if by_zone:
        print("\n---- per-zone ingress->node leg (larger = farther from ingress / more detour) ----")
        for zone in sorted(by_zone, key=lambda z: min(by_zone[z])):
            v = by_zone[zone]
            print(f"{zone:<16} n={len(v):>2}  leg_min={min(v)}ms  "
                  f"median={round(statistics.median(v), 1)}ms")

vantage point: local egress IP = 134.102.206.230 (your EU network)
you -> ingress (ebim-jupyter.aicor.dev) baseline RTT: min=25.8ms  p50=26.6ms

hitting api/status via the proxy x 30, taking min:

  #  node         zone               n      min     p95  leg~node
  0  l4-vm-290    europe-west4-c    30     27.6    30.5       1.8
  1  l4-vm-288    europe-west4-c    30     28.2    31.9       2.4
  2  l4-vm-282    us-central1-c     30    132.3   135.8     106.5
  3  l4-vm-289    europe-west4-c    30     26.9    29.1       1.1
  4  l4-vm-284    europe-west4-c    30     28.8    31.0       3.0
  5  l4-vm-288    europe-west4-c    30     29.9    34.6       4.1
  6  l4-vm-290    europe-west4-c    30     28.6    31.4       2.8
  7  l4-vm-289    europe-west4-c    30     29.2    32.9       3.4
  8  l4-vm-282    us-central1-c     30    131.8   135.1     106.0
  9  l4-vm-284    europe-west4-c    30     29.7    34.5       3.9
 10  l4-vm-281    europe-west4-c    30     31.3    34.1       5.5
 11  l4-vm-

In [8]:
# ===== latency (WebSocket ping/pong, same transport as VNC) =====
# for each pod, open a wss long-lived connection to /api/events/subscribe (via the same ebim-jupyter ingress),
# then measure a single round-trip with WebSocket-level ping/pong — the RTT closest to the VNC feel.
# note: it still passes through the US proxy (unavoidable, like VNC); it measures the real end-to-end RTT you experience;
# "bypassing the proxy to reach the pod directly" is simply unreachable from the client, so it cannot be measured.
try:
    import websocket          # pip install websocket-client
    from websocket import ABNF
except ImportError:
    print("needs websocket-client:  pip install websocket-client")
else:
    import ssl, statistics

    WS_PINGS   = 30
    WS_TIMEOUT = 10

    def ws_rtt(r):
        base = r["server_url"]
        tok  = r.get("token")
        url  = (base.replace("https://", "wss://").replace("http://", "ws://").rstrip("/")
                + "/api/events/subscribe")
        if tok:
            url += f"?token={tok}"
        header = [f"Authorization: token {tok}"] if tok else []
        sslopt = {"cert_reqs": ssl.CERT_NONE} if not VERIFY_TLS else {}
        xs, ws = [], None
        try:
            ws = websocket.create_connection(url, header=header, sslopt=sslopt,
                                             timeout=WS_TIMEOUT)
            for _ in range(WS_PINGS):
                t0 = time.perf_counter()
                ws.ping(b"p")
                deadline = t0 + WS_TIMEOUT
                while True:                       # read until PONG, ignoring intervening event data frames
                    op, _ = ws.recv_data(control_frame=True)
                    if op == ABNF.OPCODE_PONG:
                        xs.append((time.perf_counter() - t0) * 1000)
                        break
                    if time.perf_counter() > deadline:
                        break
        except Exception as e:
            return None, repr(e)[:90]
        finally:
            if ws:
                try: ws.close()
                except Exception: pass
        return sorted(xs), None

    ready = [r for r in results if r.get("ok")]
    print(f"WebSocket ping/pong x {WS_PINGS}  ->  /api/events/subscribe (via ingress to pod)\n")
    print(f"{'#':>3}  {'node':<12} {'zone':<16} {'n':>3}  {'min':>7} {'p50':>7} {'p95':>7}  err")
    ws_rows = []
    for r in sorted(ready, key=lambda x: x["idx"]):
        xs, err = ws_rtt(r)
        inst = node_instance(r["node"]) or "-"
        zone = node_zone(r["node"])
        if xs:
            n = len(xs)
            pk = lambda q: xs[min(n - 1, int(n * q))]
            mn = round(xs[0], 1)
            ws_rows.append({"zone": zone, "min": mn})
            print(f"{r['idx']:>3}  {inst:<12} {zone:<16} {n:>3}  "
                  f"{mn:>7} {round(pk(0.5),1):>7} {round(pk(0.95),1):>7}")
        else:
            print(f"{r['idx']:>3}  {inst:<12} {zone:<16}   0  "
                  f"{'-':>7} {'-':>7} {'-':>7}  {err or 'no pong'}")

    by_zone = {}
    for row in ws_rows:
        by_zone.setdefault(row["zone"], []).append(row["min"])
    if by_zone:
        print("\n---- per-zone WebSocket RTT (min; smaller = more responsive) ----")
        for zone in sorted(by_zone, key=lambda z: min(by_zone[z])):
            v = by_zone[zone]
            print(f"{zone:<16} n={len(v):>2}  min={min(v)}ms  "
                  f"median={round(statistics.median(v), 1)}ms")

WebSocket ping/pong x 30  ->  /api/events/subscribe (via ingress to pod)

  #  node         zone               n      min     p50     p95  err
  0  l4-vm-290    europe-west4-c    30     22.3    22.7    24.2
  1  l4-vm-288    europe-west4-c    30     23.3    23.7    24.6
  2  l4-vm-282    us-central1-c     30    125.4   125.8   127.2
  3  l4-vm-289    europe-west4-c    30     23.0    23.6    26.2
  4  l4-vm-284    europe-west4-c    30     21.0    21.4    22.6
  5  l4-vm-288    europe-west4-c    30     23.0    23.5    24.8
  6  l4-vm-290    europe-west4-c    30     23.0    23.4    23.9
  7  l4-vm-289    europe-west4-c    30     23.0    23.7    24.6
  8  l4-vm-282    us-central1-c     30    125.3   125.6   127.4
  9  l4-vm-284    europe-west4-c    30     23.5    23.9    25.4
 10  l4-vm-281    europe-west4-c    30     21.5    22.0    23.9
 11  l4-vm-281    europe-west4-c    30     21.4    21.7    23.3
 12  l4-vm-286    us-east1-b        30    122.4   123.0   124.2
 13  l4-vm-286    us-east

## Notes

- **Fully anonymous**: the public BinderHub build API needs no login; leave `AUTH_TOKEN` as `None`.
- **No TLS verification**: `VERIFY_TLS=False` passes `verify=False` to `requests` and silences the warning.
- **Rate limiting is BinderHub's own, per client IP — not GitHub's**: `Rate limit exceeded. Try again in 3600 seconds.` is raised by BinderHub's `RateLimiter` (`web:1964`), counted per **caller's real IP**, default **limit=10 / period_seconds=3600** (10 launches per source IP per hour). The IP in parentheses in the log is the client IP; different IPs have independent counters — that's why "switching machines" avoids the limit. Relax it server-side via `c.RateLimiter.limit` / `c.RateLimiter.period_seconds` (see `values-gc-overrides.yaml`).
- **Parallel launches**: `CONCURRENCY` controls how many run at once (parallel once the RateLimiter is relaxed). If you still hit 429s, raise `c.RateLimiter.limit` further or lower `CONCURRENCY`.
- **Where the node name comes from**: the `launching`-phase k8s event `Successfully assigned <ns>/<pod> to <node>`, parsed by `SCHED_RE`. When an existing server is reused (no new pod) there is no such event and `node` is `None`.
- **How latency is measured**: node static IPs are egress-only (not pingable), and the whole cluster has a single public ingress (`ebim-jupyter`) with the proxy in the US, so this is the only measurable path (VNC uses it too).
  - HTTP decomposition: base = you→ingress, pod = you→ingress→pod, leg = pod−base ≈ the ingress→node hop.
  - WebSocket form: ping/pong on `/api/events/subscribe`, same transport as VNC and closest to the real feel.
  - The main latency comes from the **ingress being in the US**; fixing it for good means putting the ingress in Europe or doing geo-routing — the client-side measurement method can't change that.
- **First vs later**: the first time really builds the image (slow); afterwards it hits the cache and launch is fast.
- Set the last segment of `SPEC` to a full commit SHA to pin an exact commit and avoid branch-cache ambiguity.